<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/pulling_time_serie_indices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Serie de Tiempo de Vegetacion sobre el area de la laguna de *Barcelona de Indias*

# Autenticacion

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# Parametros generales del analisis
# Sentinel-1 tiene disponibilidad en esta zona desde 2017; por eso no se usa una fecha artificial anterior.
FECHA_INICIO_ANALISIS = '2017-01-01'
FECHA_FIN_ANALISIS = '2026-05-13'

# La mascara de agua se construye con imagenes Sentinel-2 recientes para obtener una referencia espacial fija.
# Esta decision permite comparar todas las fechas contra la misma superficie de laguna.
FECHA_INICIO_MASCARA = '2025-01-01'
FECHA_FIN_MASCARA = '2025-12-31'

UMBRAL_MNDWI_AGUA = 0.05
UMBRAL_VV_VEGETACION = -16
UMBRAL_VH_VEGETACION = -22
ESCALA_ANALISIS = 10

# Ejecutar Authenticate
ee.Authenticate(auth_mode='notebook')
ee.Initialize()

# Procesamiento & Carga

Primero definimos el area de estudio de la laguna

### roi & agua (mndwi)

In [ ]:
# ROI de la laguna
roi = ee.Geometry.Polygon([
    [
        [-75.476052, 10.517524], # Longitud primero, luego Latitud
        [-75.476117, 10.518747],
        [-75.473158,10.519223 ],
        [-75.470516, 10.525108],
        [-75.469572, 10.524876],
        [-75.471686, 10.518916],
        [-75.468394, 10.517219],
        [-75.468952, 10.516459],
        [-75.476052, 10.517524]  # Cierre del polígono
    ]
])

Ahora obtenemos la mascara de la laguna sobre el ROI elegido con una fecha de inicio y fecha fin sobre la cual obtendremos la mediana de esos pixeles para tener mas claramente esa zona de agua

In [ ]:
def mask_s2_clouds(image):
    """
    Realiza el enmascaramiento de nubes y cirros para Sentinel-2 SR.

    Proceso:
    1. Identifica nubes y cirros usando la banda QA60.
    2. Escala los valores de reflectancia de DN a valores [0, 1].
    3. Preserva las propiedades temporales de la imagen original.

    Args:
        image (ee.Image): Imagen de la colección COPERNICUS/S2_SR_HARMONIZED.

    Returns:
        ee.Image: Imagen enmascarada y escalada.
    """
    image = ee.Image(image)
    qa = image.select('QA60')

    # Definición de bits para nubes (10) y cirros (11)
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Crear máscara: ambos bits deben ser 0 para indicar cielo despejado
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
            .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))

    return (image.updateMask(mask)
            .divide(10000)
            .copyProperties(image, ['system:time_start']))


def obtener_mascara_laguna(roi, fecha_inicio=FECHA_INICIO_MASCARA,
                           fecha_fin=FECHA_FIN_MASCARA):
    """
    Genera una máscara binaria representativa del espejo de agua estable.

    Utiliza el índice MNDWI sobre un compuesto de mediana temporal para
    minimizar el ruido de nubes residuales y sombras.

    Args:
        roi (ee.Geometry): Polígono que delimita la laguna.
        fecha_inicio (str): Fecha de inicio para el compuesto de referencia.
        fecha_fin (str): Fecha de fin para el compuesto de referencia.

    Returns:
        ee.Image: Máscara booleana (1: agua, 0: otros).
    """
    # 1. Colección Sentinel-2 con pre-procesamiento
    s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                     .filterBounds(roi)
                     .filterDate(fecha_inicio, fecha_fin)
                     .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                     .map(mask_s2_clouds))

    # 2. Compuesto de mediana (ideal para eliminar valores atípicos y nubes)
    composite = s2_collection.median()

    # 3. Cálculo de MNDWI: (Green - SWIR1) / (Green + SWIR1)
    # Se usan bandas B3 y B11 respectivamente
    mndwi = composite.normalizedDifference(['B3', 'B11']).rename('MNDWI')

    # 4. Segmentación usando la constante UMBRAL_MNDWI_AGUA
    # Nota: Tu valor previo 0.000001 era muy bajo; usamos el definido arriba (0.05)
    agua = mndwi.gt(UMBRAL_MNDWI_AGUA)

    # 5. Post-procesamiento morfológico (Operación de Cierre)
    # Focal_max seguido de focal_min para eliminar huecos internos y consolidar bordes
    agua_limpia = (agua
                   .focal_max(radius=1, units='pixels')
                   .focal_min(radius=1, units='pixels')
                   .rename('agua'))

    return agua_limpia


# --- EJECUCIÓN ---
# Se genera la máscara fija que servirá de base para todo el análisis temporal
mascara_fija = obtener_mascara_laguna(roi)

In [ ]:
# 1. Crear la instancia del mapa
Map = geemap.Map()

# 2. Centrar el mapa
Map.centerObject(roi, 16)

# 3. Añadir la Máscara de Agua (solo lo que es agua)
Map.addLayer(mascara_fija.selfMask(), {'palette': ['0000FF']}, 'Máscara de Agua (Laguna)')

# 4. Añadir el ROI con estilo (Corregido)
# Convertimos el roi (Geometry) a FeatureCollection para usar .style()
roi_style = {'color': 'red', 'fillColor': '00000000', 'width': 2}
Map.addLayer(ee.FeatureCollection([ee.Feature(roi)]).style(**roi_style), {}, 'Límite del ROI')

# 5. Mostrar el mapa
Map

Ahora veamos a cuanto equivale en m2 el area de la laguna

In [ ]:
# =============================================================================
# CÁLCULO DEL ÁREA TOTAL DE LA LAGUNA (MÁSCARA FIJA)
# =============================================================================

def calcular_area_total_mascara(mascara, region):
    """
    Calcula el área total en metros cuadrados de una máscara binaria.

    Args:
        mascara (ee.Image): Imagen binaria (1 para agua).
        region (ee.Geometry): Área de interés (ROI).

    Returns:
        float: Área total en metros cuadrados.
    """
    # Multiplicamos la máscara (1s) por el área real de cada píxel
    area_imagen = mascara.multiply(ee.Image.pixelArea())

    # Reducción de la región para sumar las áreas de los píxeles con valor 1
    estadisticas = area_imagen.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=ESCALA_ANALISIS,
        maxPixels=1e9
    )

    # Retornamos el valor numérico (get devuelve un objeto de EE, .getInfo() lo trae local)
    return estadisticas.get('agua').getInfo()

# Ejecución del cálculo
area_total_laguna_m2 = calcular_area_total_mascara(mascara_fija, roi)

print(f"Área total del espejo de agua (Mascara Fija): {area_total_laguna_m2:,.2f} m²")

Ahora que ya tenemos la mascara fija del agua, ahora vamos a hallar la vegetacion sobre esa mascara de agua. Para esto usaremos varios indices:

1. VV-VH *SENTINEL-1*
2. FAI (Floating Algae Index) *SENTINEL-2*
3. B4 (Red) *SENTINEL-2*
4. B8 (NIR) *SENTINEL-2*
5. B11 (SWIR) *SENTINEL-2*
6. NDAVI (Normalized Difference Aquatic Vegetation Index) *SENTINEL-2*

### Indices Opticos

Sentinel-2 nos permite usar la respuesta de la clorofila. El FAI es excelente para detectar vegetación flotante (como el buchón o algas) porque resalta objetos que reflejan mucho en el infrarrojo cercano (NIR) en comparación con el agua circundante.

In [ ]:
def agregar_indices_opticos(image):
    """
    Calcula índices de vegetación acuática (NDAVI y FAI) para Sentinel-2.
    """
    image = ee.Image(image)

    # 1. NDAVI: (NIR - Blue) / (NIR + Blue)
    # Útil para diferenciar plantas acuáticas de agua abierta
    ndavi = image.normalizedDifference(['B8', 'B2']).rename('NDAVI')

    # 2. FAI (Floating Algae Index)
    # Resalta la reflectancia del NIR sobre una línea base entre Red y SWIR
    red = image.select('B4')
    nir = image.select('B8')
    swir = image.select('B11')

    # Constante de corrección de pendiente para Sentinel-2 (~0.187)
    fai = nir.subtract(
        red.add(
            swir.subtract(red).multiply(0.187)
        )
    ).rename('FAI')

    return image.addBands([ndavi, fai])

### Forma Radar

En el radar, el agua tranquila actúa como un espejo (se ve negra), pero la vegetación que sobresale del agua causa un fenómeno llamado Double-Bounce o dispersión volumétrica, lo que aumenta mucho los valores de VV y VH.

In [ ]:
def procesar_radar_vegetacion(image):
    """
    Prepara las bandas de radar para la detección de vegetación.
    """
    # Calculamos la diferencia VV-VH que a veces resalta estructuras verticales
    diff = image.select('VV').subtract(image.select('VH')).rename('VV_VH_diff')
    return image.addBands(diff)

### Integracion

Para hallar la vegetación sobre el agua, debemos aplicar la mascara_fija a cada colección. Esto asegura que no estemos contando vegetación de la orilla (terrestre).

In [ ]:
# =============================================================================
# PRE-PROCESAMIENTO DE COLECCIONES SATELITALES
# =============================================================================

# --- 1. PROCESAMIENTO SENTINEL-2 (DATOS ÓPTICOS) ---
# Esta sección filtra la colección por espacio y tiempo, elimina nubes,
# calcula índices de vegetación acuática y confina el análisis al espejo de agua.
coleccion_s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(FECHA_INICIO_ANALISIS, FECHA_FIN_ANALISIS)
    # Aplicación de máscara de nubes y escalado de reflectancia
    .map(mask_s2_clouds)
    # Cálculo de índices específicos para vegetación acuática (FAI, NDAVI)
    .map(agregar_indices_opticos)
    # Recorte espacial usando la máscara de agua fija para aislar la laguna
    .map(lambda img: img.updateMask(mascara_fija))
)

# --- 2. PROCESAMIENTO SENTINEL-1 (DATOS DE RADAR) ---
coleccion_s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(roi)
    .filterDate(FECHA_INICIO_ANALISIS, FECHA_FIN_ANALISIS)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    # --- FILTRO CRÍTICO ---
    # Seleccionamos solo imágenes que tengan tanto VV como VH
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    # ----------------------
    .map(procesar_radar_vegetacion)
    .map(lambda img: img.updateMask(mascara_fija))
)

Ahora que ya tenemos las colecciones, ahora sí queremos al final es sacar estadisticas del area de vegetacion por cada uno de los indices

Para los indices simples

In [ ]:
def calcular_area_vegetacion(coleccion, banda_indice, umbral, nombre_metrica):
    """
    Calcula el área de vegetación detectada por encima de un umbral específico.
    Utiliza una conversión de fecha numérica para evitar errores de parseo.
    """
    def extraer_datos(image):
        # 1. Crear máscara binaria basada en el umbral
        # Usamos un casting explícito para evitar errores de tipo
        binaria = ee.Image(image).select(banda_indice).gt(umbral)

        # 2. Calcular el área de los píxeles (1 * área_píxel)
        area_imagen = binaria.multiply(ee.Image.pixelArea())

        # 3. Reducir la región para obtener la suma total del área
        estadisticas = area_imagen.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=ESCALA_ANALISIS,
            maxPixels=1e9
        )

        # 4. Obtener la fecha como timestamp de milisegundos (más seguro)
        return ee.Feature(None, {
            'timestamp': image.date().millis(), # Usamos milisegundos en vez de string
            nombre_metrica: estadisticas.get(banda_indice),
            'id_imagen': image.id()
        })

    # Convertir la colección a una lista de características
    # El filtrado de nulos asegura que no procesemos imágenes vacías
    features_list = coleccion.map(extraer_datos).filter(ee.Filter.notNull([nombre_metrica]))

    # Obtener datos de la nube
    info = features_list.getInfo()['features']
    datos_lista = [f['properties'] for f in info]

    if not datos_lista:
        print(f"Advertencia: No se encontraron datos para {nombre_metrica}")
        return pd.DataFrame()

    df = pd.DataFrame(datos_lista)

    # Convertir el timestamp de milisegundos a objetos datetime de Pandas
    df['fecha'] = pd.to_datetime(df['timestamp'], unit='ms')

    # Limpieza: eliminar columna auxiliar de timestamp
    df = df.drop(columns=['timestamp'])

    return df

Para la logica de Sentinel-1 combinada de VV-VH

In [ ]:
def extraer_datos_radar_robusto(image):
    """
    Detecta vegetación usando una regla lógica combinada VV y VH.
    Calcula áreas y porcentajes de cobertura basados en la máscara fija.
    """
    # 1. Preparar imagen y bandas
    # Usamos la imagen ya procesada (con la banda diff y la máscara fija aplicada)
    vv = image.select('VV')
    vh = image.select('VH')

    # 2. Aplicar Regla Lógica: Debe cumplir ambos umbrales para ser vegetación
    # Esto reduce significativamente los falsos positivos por ruido u oleaje
    vegetacion = vv.gt(UMBRAL_VV_VEGETACION).And(vh.gt(UMBRAL_VH_VEGETACION)).rename('veg')

    # 3. Calcular área de vegetación detectada (m²)
    area_veg = vegetacion.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=ESCALA_ANALISIS,
        maxPixels=1e9
    ).get('veg')

    # 4. Obtener el área total del lago (usando nuestra constante ya calculada)
    # Nota: Usamos el objeto ee.Number para cálculos en el servidor
    area_lago_ee = ee.Number(area_total_laguna_m2)
    area_veg_ee = ee.Number(area_veg)

    # 5. Calcular porcentaje de cobertura
    porcentaje = area_veg_ee.divide(area_lago_ee).multiply(100)

    # 6. Retornar Feature con metadatos y métricas
    return ee.Feature(None, {
        'fecha': image.date().format('YYYY-MM-dd'),
        'timestamp': image.date().millis(),
        'id_imagen': image.get('system:index'),
        'area_m2_radar_vv': area_veg_ee,  # Nombre corregido
        'pct_coverage_radar': porcentaje
    })

# --- EJECUCIÓN DEL PROCESAMIENTO ---
# Mapeamos la nueva función sobre la colección de radar ya filtrada
datos_radar_robustis = coleccion_s1.map(extraer_datos_radar_robusto)

# Convertir a DataFrame de Pandas
info_radar = datos_radar_robustis.getInfo()['features']
df_radar_combined = pd.DataFrame([f['properties'] for f in info_radar])
df_radar_combined['fecha'] = pd.to_datetime(df_radar_combined['timestamp'], unit='ms')

In [ ]:
# =============================================================================
# EJECUCIÓN MULTI-ÍNDICE (SENTINEL-1 & SENTINEL-2)
# =============================================================================

# --- A. Procesar índices estándar (Sentinel-2 y bandas simples) ---
configuracion_indices = {
    'area_m2_fai': (coleccion_s2, 'FAI', 0.16),
    'area_m2_ndavi': (coleccion_s2, 'NDAVI', 0.4),
    'area_m2_red_b4': (coleccion_s2, 'B4', 0.58),
    'area_m2_nir_b8': (coleccion_s2, 'B8', 0.66),
    'area_m2_swir_b11': (coleccion_s2, 'B11', 0.65)
}

resultados_dfs = {}

for metrica, params in configuracion_indices.items():
    print(f"Procesando índice óptico: {metrica}...")
    resultados_dfs[metrica] = calcular_area_vegetacion(params[0], params[1], params[2], metrica)

# --- B. Procesar Radar con la Lógica Robusta (VV & VH) ---
print("Procesando Radar Robusto (VV-VH Logic)...")
datos_radar_pass = coleccion_s1.map(extraer_datos_radar_robusto)
info_radar = datos_radar_pass.getInfo()['features']

df_radar_robusto = pd.DataFrame([f['properties'] for f in info_radar])
df_radar_robusto['fecha'] = pd.to_datetime(df_radar_robusto['timestamp'], unit='ms')
df_radar_robusto = df_radar_robusto.drop(columns=['timestamp'])

# Agregamos el resultado robusto al diccionario principal
resultados_dfs['area_m2_radar_vv'] = df_radar_robusto

Ahora unimos todo

In [ ]:
# =============================================================================
# CONSOLIDACIÓN MAESTRA DE TODOS LOS ÍNDICES (MERGE)
# =============================================================================

# Lista de todas las métricas procesadas
lista_metricas = list(resultados_dfs.keys())

# Empezamos el merge con el primer DataFrame
df_final = resultados_dfs[lista_metricas[0]]

for i in range(1, len(lista_metricas)):
    metrica_actual = lista_metricas[i]
    # Hacemos el merge sobre 'fecha' asegurando que no se pierdan días (outer join)
    df_final = pd.merge(
        df_final,
        resultados_dfs[metrica_actual][['fecha', metrica_actual]],
        on='fecha',
        how='outer'
    )

# Ordenar cronológicamente
df_final = df_final.sort_values('fecha').reset_index(drop=True)

# Limpiar valores que excedan el área total por errores de píxel (como el de 59k)
columnas_area = [c for c in df_final.columns if 'area_m2' in c]
for col in columnas_area:
    df_final[col] = df_final[col].clip(upper=area_total_laguna_m2)

print("¡Procesamiento y consolidación completados!")

In [ ]:
df_final

# Visualizaciones

## Series por Indice

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- PROFESSIONAL PLOT CONFIGURATION ---
# Increased font sizes for academic publication standards
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'serif',
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12
})

# --- EXACT SELECTION OF THE 6 REQUESTED INDICES ---
indices_solicitados = [
    'area_m2_radar_vv',
    'area_m2_fai',
    'area_m2_red_b4',
    'area_m2_nir_b8',
    'area_m2_swir_b11',
    'area_m2_ndavi'
]

# Mapping technical column names to English Paper Titles
nombres_paper = {
    'area_m2_radar_vv': 'Sentinel-1 (VV-VH Polarization)',
    'area_m2_fai': 'Floating Algae Index (FAI)',
    'area_m2_red_b4': 'B4 (Red Band) Reflectance',
    'area_m2_nir_b8': 'B8 (NIR Band) Reflectance',
    'area_m2_swir_b11': 'B11 (SWIR Band) Reflectance',
    'area_m2_ndavi': 'Normalized Difference Aquatic Vegetation Index (NDAVI)'
}

# Filtering available columns in the final DataFrame
columnas_a_graficar = [col for col in indices_solicitados if col in df_final.columns]
n_graficos = len(columnas_a_graficar)

# Creating the multi-panel figure
fig, axes = plt.subplots(n_graficos, 1, figsize=(15, 5 * n_graficos), sharex=True)

# Ensure axes is a list even for a single plot
if n_graficos == 1:
    axes = [axes]

for i, col in enumerate(columnas_a_graficar):
    # Dropping NaNs to ensure continuous lines for each specific sensor
    df_plot = df_final[['fecha', col]].dropna()

    axes[i].plot(
        df_plot['fecha'],
        df_plot[col],
        linewidth=2,
        color='#1B4F72',
        markersize=4,
        label='Calculated Area'
    )

    # Titles and Labels in English
    axes[i].set_title(f'Temporal Evolution: {nombres_paper.get(col, col)}', loc='left', fontweight='bold')
    axes[i].set_ylabel('Vegetation Area ($m^2$)', labelpad=10)
    axes[i].grid(True, linestyle='--', alpha=0.6)

    # X-Axis Configuration (Horizontal years, rotation=0)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(axes[i].get_xticklabels(), rotation=0)

# Common X-axis label
axes[-1].set_xlabel('Observation Date (Year)', labelpad=20)

plt.tight_layout()

# --- EXPORTING SECTION (COMMENTED AS REQUESTED) ---
# plt.savefig('lagoon_vegetation_6_indices.pdf', bbox_inches='tight')
# plt.savefig('lagoon_vegetation_6_indices_300dpi.png', dpi=300, bbox_inches='tight')

plt.show()

print(f"Successfully generated {n_graficos} time series plots in English.")

In [ ]:
# --- PROFESSIONAL PLOT CONFIGURATION (ACADEMIC STANDARDS) ---
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'serif',
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12
})

# --- SELECTION OF THE 6 INDICES ---
indices_solicitados = [
    'area_m2_radar_vv',
    'area_m2_fai',
    'area_m2_red_b4',
    'area_m2_nir_b8',
    'area_m2_swir_b11',
    'area_m2_ndavi'
]

# Mapping for English Paper Titles
nombres_paper = {
    'area_m2_radar_vv': 'Sentinel-1 (VV-VH Combined)',
    'area_m2_fai': 'Floating Algae Index (FAI)',
    'area_m2_red_b4': 'B4 (Red Band)',
    'area_m2_nir_b8': 'B8 (NIR Band)',
    'area_m2_swir_b11': 'B11 (SWIR Band)',
    'area_m2_ndavi': 'Aquatic Vegetation Index (NDAVI)'
}

# Filtering available columns
columnas_a_graficar = [col for col in indices_solicitados if col in df_final.columns]
n_graficos = len(columnas_a_graficar)

# Creating the figure
fig, axes = plt.subplots(n_graficos, 1, figsize=(15, 5 * n_graficos), sharex=True)

if n_graficos == 1:
    axes = [axes]

for i, col in enumerate(columnas_a_graficar):
    # Dropping NaNs for a clean time series
    df_plot = df_final[['fecha', col]].dropna()

    # --- PERCENTAGE CALCULATION ---
    # We calculate the percentage of coverage relative to the total lagoon area
    percentage_coverage = (df_plot[col] / area_total_laguna_m2) * 100

    axes[i].plot(
        df_plot['fecha'],
        percentage_coverage,
        linewidth=2,
        color='#1B4F72',
        marker='o',
        markersize=3,
        label='Coverage Percentage'
    )

    # Customizing each subplot in English
    axes[i].set_title(f'Vegetation Coverage: {nombres_paper.get(col, col)}', loc='left', fontweight='bold')
    axes[i].set_ylabel('Coverage (%)', labelpad=10)
    axes[i].set_ylim(0, 105)  # Set limit to 105% to see the top clearly
    axes[i].grid(True, linestyle='--', alpha=0.6)

    # X-Axis format
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(axes[i].get_xticklabels(), rotation=0)

# Common X-axis label
axes[-1].set_xlabel('Observation Date (Year)', labelpad=20)

plt.tight_layout()

# --- EXPORTING SECTION (COMMENTED) ---
# plt.savefig('vegetation_percentage_coverage_6_indices.pdf', bbox_inches='tight')
# plt.savefig('vegetation_percentage_coverage_300dpi.png', dpi=300, bbox_inches='tight')

plt.show()

print(f"Percentage coverage plots successfully generated for {n_graficos} indices.")

## Casos ejemplo de VV-VH & NDAVI

In [ ]:
# =============================================================================
# DETECTION OF VEGETATION PEAKS (MAXIMA)
# =============================================================================

def get_peak_data(df, column_name):
    """
    Identifies the date and maximum area for a specific index.

    Args:
        df (pd.DataFrame): Consolidated dataframe with all indices.
        column_name (str): Column to analyze (e.g., 'area_m2_radar_vv').

    Returns:
        pd.Series: Row containing the peak information.
    """
    # Find the index of the maximum value in the specified column
    # We drop NaNs to ensure we are looking at real observations
    peak_idx = df[column_name].idxmax()
    return df.loc[peak_idx]

In [ ]:
# =============================================================================
# DETECTION OF VEGETATION PEAKS (RADAR, NDAVI & FAI)
# =============================================================================

# 1. Identify peaks for the requested indices (Now including FAI)
peak_radar = get_peak_data(df_final, 'area_m2_radar_vv')
peak_ndavi = get_peak_data(df_final, 'area_m2_ndavi')
peak_fai   = get_peak_data(df_final, 'area_m2_fai')

# 2. Format and display the results in English
print("--- VEGETATION PEAK DETECTION RESULTS ---")

# Sentinel-1 Radar Peak
print(f"\n[SENTINEL-1 RADAR PEAK]")
print(f"Date: {peak_radar['fecha'].strftime('%Y-%m-%d')}")
print(f"Maximum Area: {peak_radar['area_m2_radar_vv']:,.2f} m²")
print(f"Coverage: {(peak_radar['area_m2_radar_vv']/area_total_laguna_m2)*100:.2f}%")
print(f"Image ID: {peak_radar['id_imagen']}")

# NDAVI Optical Peak
print(f"\n[NDAVI OPTICAL PEAK]")
print(f"Date: {peak_ndavi['fecha'].strftime('%Y-%m-%d')}")
print(f"Maximum Area: {peak_ndavi['area_m2_ndavi']:,.2f} m²")
print(f"Coverage: {(peak_ndavi['area_m2_ndavi']/area_total_laguna_m2)*100:.2f}%")
print(f"Image ID: {peak_ndavi['id_imagen']}")

# FAI Optical Peak
print(f"\n[FAI OPTICAL PEAK]")
print(f"Date: {peak_fai['fecha'].strftime('%Y-%m-%d')}")
print(f"Maximum Area: {peak_fai['area_m2_fai']:,.2f} m²")
print(f"Coverage: {(peak_fai['area_m2_fai']/area_total_laguna_m2)*100:.2f}%")
print(f"Image ID: {peak_fai['id_imagen']}")

# 3. Store dates for later mapping or visualization
radar_peak_date = peak_radar['fecha'].strftime('%Y-%m-%d')
ndavi_peak_date = peak_ndavi['fecha'].strftime('%Y-%m-%d')
fai_peak_date   = peak_fai['fecha'].strftime('%Y-%m-%d')

In [ ]:
def generar_mapa_pico_estilizado(id_imagen, fecha_str, area, cobertura, tipo="radar"):
    Map = geemap.Map()
    Map.centerObject(roi, 16)

    # 1. Obtener la imagen según el tipo
    if tipo == "radar":
        # Si el ID es nan, buscamos por fecha en la colección de radar
        if str(id_imagen).lower() == 'nan':
            imagen = coleccion_s1.filterDate(fecha_str, ee.Date(fecha_str).advance(1, 'day')).first()
        else:
            imagen = coleccion_s1.filter(ee.Filter.eq('system:index', id_imagen)).first()

        # Procesamiento Radar (VV/VH)
        imagen_clip = imagen.clip(roi).updateMask(mascara_fija)
        capa_veg = imagen_clip.select('VV').gt(UMBRAL_VV_VEGETACION).And(imagen_clip.select('VH').gt(UMBRAL_VH_VEGETACION))
        nombre_capa = "Radar Peak"

    else:
        # Procesamiento Óptico (Sentinel-2)
        imagen = ee.Image(f'COPERNICUS/S2_SR_HARMONIZED/{id_imagen}')
        imagen_con_indices = agregar_indices_opticos(imagen).clip(roi).updateMask(mascara_fija)

        if tipo == "ndavi":
            capa_veg = imagen_con_indices.select('NDAVI').gt(0.3)
            nombre_capa = "NDAVI Peak"
        else: # FAI
            capa_veg = imagen_con_indices.select('FAI').gt(0.05)
            nombre_capa = "FAI Peak"

    # 2. Estilización visual (como en tu foto)
    # Fondo azul oscuro para la laguna
    Map.addLayer(mascara_fija.selfMask(), {'palette': ['#00008B']}, 'Capa de Agua (Laguna)')

    # Vegetación en verde brillante (selfMask para no pintar el fondo)
    Map.addLayer(capa_veg.selfMask(), {'palette': ['#00FF00']}, f'Vegetación {nombre_capa}')

    print(f"--- {nombre_capa} ---")
    print(f"Fecha: {fecha_str} | Área: {area:,.2f} m² | Cobertura: {cobertura:.2f}%")

    return Map

1 de los Picos HIstoricos de VV-VH

In [ ]:
mapa_radar = generar_mapa_pico_estilizado(
    id_imagen='nan',
    fecha_str='2017-05-14',
    area=14992.57,
    cobertura=25.46,
    tipo="radar"
)
mapa_radar

In [ ]:
mapa_ndavi = generar_mapa_pico_estilizado(
    id_imagen='20240220T153621_20240220T153623_T18PVS',
    fecha_str='2024-02-20',
    area=34525.33,
    cobertura=58.62,
    tipo="ndavi"
)
mapa_ndavi

Aqui mostramos otros 2 picos diferentes de RADAR y NDAVI

In [ ]:
# Definimos las fechas que ya usamos para excluirlas
fechas_excluir = ['2017-05-14', '2024-02-20', '2023-10-08']

# Filtramos el dataframe para obtener datos nuevos
df_filtrado = df_final[~df_final['fecha'].dt.strftime('%Y-%m-%d').isin(fechas_excluir)]

# 1. Identificar nuevos picos (los más altos después de los anteriores)
new_peak_radar = get_peak_data(df_filtrado, 'area_m2_radar_vv')
new_peak_ndavi = get_peak_data(df_filtrado, 'area_m2_ndavi')

# 2. Formatear y mostrar los resultados
print("--- NEW HISTORICAL PEAK DETECTION (SECONDARY PEAKS) ---")

# Nuevo Pico Radar
print(f"\n[NEW SENTINEL-1 RADAR PEAK]")
print(f"Date: {new_peak_radar['fecha'].strftime('%Y-%m-%d')}")
print(f"Maximum Area: {new_peak_radar['area_m2_radar_vv']:,.2f} m²")
print(f"Coverage: {(new_peak_radar['area_m2_radar_vv']/area_total_laguna_m2)*100:.2f}%")
print(f"Image ID: {new_peak_radar['id_imagen']}")

# Nuevo Pico NDAVI
print(f"\n[NEW NDAVI OPTICAL PEAK]")
print(f"Date: {new_peak_ndavi['fecha'].strftime('%Y-%m-%d')}")
print(f"Maximum Area: {new_peak_ndavi['area_m2_ndavi']:,.2f} m²")
print(f"Coverage: {(new_peak_ndavi['area_m2_ndavi']/area_total_laguna_m2)*100:.2f}%")
print(f"Image ID: {new_peak_ndavi['id_imagen']}")

In [ ]:
# --- MAPA PARA EL NUEVO PICO DE RADAR ---
mapa_nuevo_radar = generar_mapa_pico_estilizado(
    id_imagen=str(new_peak_radar['id_imagen']),
    fecha_str=new_peak_radar['fecha'].strftime('%Y-%m-%d'),
    area=new_peak_radar['area_m2_radar_vv'],
    cobertura=(new_peak_radar['area_m2_radar_vv']/area_total_laguna_m2)*100,
    tipo="radar"
)
mapa_nuevo_radar

In [ ]:
# --- MAPA PARA EL NUEVO PICO DE NDAVI ---
mapa_nuevo_ndavi = generar_mapa_pico_estilizado(
    id_imagen=str(new_peak_ndavi['id_imagen']),
    fecha_str=new_peak_ndavi['fecha'].strftime('%Y-%m-%d'),
    area=new_peak_ndavi['area_m2_ndavi'],
    cobertura=(new_peak_ndavi['area_m2_ndavi']/area_total_laguna_m2)*100,
    tipo="ndavi"
)
mapa_nuevo_ndavi